# 🚀 LifeMate - Zalo AI Challenge 2026 (Google Colab GPU T4)

Notebook này hướng dẫn chi tiết cách chạy pipeline Agentic Voice Assistant trên **Google Colab GPU Tesla T4 (15GB VRAM)** mà **hoàn toàn không bị CUDA Out of Memory (OOM)**.

### ⚡ Giải pháp chống tràn VRAM trên Tesla T4:
1. **4-bit NF4 Quantization (BitsAndBytes)**: Tự động kích hoạt trên GPU, giảm bộ nhớ weights từ **~9.1 GB** xuống chỉ còn **~2.8 GB** VRAM.
2. **Quản lý bộ nhớ PyTorch**: Thiết lập `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` và tự động giải phóng cache (`torch.cuda.empty_cache()`) sau mỗi lượt suy luận.
3. **Tốc độ cực nhanh**: Khoảng **2 - 4 giây / mẫu** $\rightarrow$ Chạy xong 634 mẫu `public_test` chỉ trong **~30 - 45 phút**!

## Bước 1: Kiểm tra GPU Tesla T4

In [ ]:
# Đảm bảo đã bật GPU: Menu Runtime -> Change runtime type -> T4 GPU
!nvidia-smi

## Bước 2: Kết nối Google Drive để nạp Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Bước 3: Clone Codebase từ Branch mới

In [ ]:
# Thay <URL_REPO_CỦA_BẠN> và <BRANCH_NAME> bằng repo và branch của bạn
%cd /content
!rm -rf LifeMate
!git clone -b colab-t4 <URL_REPO_CỦA_BẠN> LifeMate
%cd /content/LifeMate

## Bước 4: Cài đặt thư viện phụ thuộc (bao gồm bitsandbytes)

In [ ]:
# Cài đặt các thư viện: torch, transformers, bitsandbytes, qwen-agent, sherpa-onnx
!pip install -r requirements.txt

## Bước 5: Chạy thử nghiệm trên 2 mẫu (Sanity Check)
Kiểm tra xem pipeline có chạy mượt mà trên T4 GPU với 4-bit NF4 mà không bị lỗi OOM không.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Cập nhật đường dẫn thư mục public_test trên Google Drive của bạn
DATA_DIR = "/content/drive/MyDrive/ZAIC2026/public_test"  # <-- Sửa thành đường dẫn trên Drive của bạn

!python main.py \
    --mode public_test \
    --num-samples 2 \
    --data-dir "$DATA_DIR" \
    --load-in-4bit \
    --output test_submission.jsonl

## Bước 6: Chạy toàn bộ 634 mẫu public_test & Xuất submission.jsonl

In [ ]:
# Chạy toàn bộ tập public_test (khoảng 30 - 45 phút trên T4 GPU)
# Lưu ý: main.py có cơ chế tự động resume nếu bị ngắt kết nối giữa chừng
!python main.py \
    --mode public_test \
    --num-samples -1 \
    --data-dir "$DATA_DIR" \
    --load-in-4bit \
    --output submission.jsonl

### 💡 Gợi ý thêm: Chạy với Qwen2.5-3B-Instruct (siêu nhẹ, chỉ 3.09B params < 5B)
Nếu bạn muốn thử một mô hình khác cũng tuân thủ luật thi (< 5B), cực kỳ nhẹ và nhanh:
```bash
!python main.py --mode public_test --llm Qwen/Qwen2.5-3B-Instruct --num-samples -1 --data-dir "$DATA_DIR" --output submission_qwen2.5.jsonl
```

## Bước 7: Kiểm tra kết quả & Lưu file nộp bài vào Google Drive

In [ ]:
# Đếm số dòng (phải đủ 634 mẫu)
!wc -l submission.jsonl

# Xem 3 dòng đầu
!head -n 3 submission.jsonl

# Copy trực tiếp vào Google Drive để tải về nộp bài
!cp submission.jsonl /content/drive/MyDrive/submission_final.jsonl
print("✅ Đã sao chép submission_final.jsonl vào Google Drive thành công!")